In [1]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from shutil import copy
import numpy as np
import pandas as pd
#import tensorflow as tf
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm


from glob import glob
import psi4
from helper_CC_ML_spacial import *



  Threads set to 12 by Python driver.


# These are the machine learning features, these will be useful later

In [2]:
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

# Use the basis sets for both Psi4 and PySCF

In [3]:

basis_sets = ['STO-3G','cc-pVDZ','aug-cc-pVDZ']

# Load in the xyz coordinates of the system and then run Psi4 with the Psi4Numpy code we use for DDCC

In [7]:

with open('../check_amplitudes/diatomics/NN.xyz','r') as f:
    text=f.read()


qmol = psi4.qcdb.Molecule.from_string(text, dtype='xyz')
mol = psi4.geometry(qmol.create_psi4_string_from_molecule()+ 'symmetry c1')                

psi4.core.clean()
psi4.core.be_quiet()

psi4.set_options({'basis': basis_sets[0],
                  'scf_type':     'pk',
                  'reference':    'rohf',
                  'mp2_type':     'conv',
                  'e_convergence': 1e-8,
                  'd_convergence': 1e-8})

rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)

A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)
check=False
if check==True:
    A.t1 = np.random.rand(*A.t1.shape)
    A.t2 = np.random.rand(*A.t2.shape)

Computing RHF reference.
Cutting 2 core orbitals.

Initalizing CCSD object...

Starting AO ->  MO transformation...
Size of the ERI tensor is 0.00 GB, 10 basis functions.
(10, 10)
(8, 8)
Building initial guess...

..initialized CCSD in 0.004 seconds.



/Users/grierjones/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:293: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'SCF_TYPE': 'PK'})
/Users/grierjones/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:294: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'E_CONVERGENCE': 10e-10})
/Users/grierjones/DDLUCJ/injected_parameters/helper_CC_ML_spacial.py:295: FutureWarning: Using `psi4.set_module_options(<module>, {<key>: <val>})` instead of `psi4.set_options({<module>__<key>: <val>})` is deprecated, and as soon as 1.5 it will stop working

  psi4.set_module_options('SCF', {'D_CONVERGENCE': 10e-10})


In [8]:
A.compute_energy()



CCSD Iteration   0: CCSD correlation = -0.154646624232695   dE =  1.54647E-01   MP2
CCSD Iteration   1: CCSD correlation = -0.147891140590810   dE =  6.75548E-03   DIIS = 0
CCSD Iteration   2: CCSD correlation = -0.151291559003040   dE = -3.40042E-03   DIIS = 1
CCSD Iteration   3: CCSD correlation = -0.152315840580268   dE = -1.02428E-03   DIIS = 2
CCSD Iteration   4: CCSD correlation = -0.153350720965911   dE = -1.03488E-03   DIIS = 3
CCSD Iteration   5: CCSD correlation = -0.153514987399625   dE = -1.64266E-04   DIIS = 4
CCSD Iteration   6: CCSD correlation = -0.153509819978286   dE =  5.16742E-06   DIIS = 5
CCSD Iteration   7: CCSD correlation = -0.153512855321501   dE = -3.03534E-06   DIIS = 6
CCSD Iteration   8: CCSD correlation = -0.153512629622594   dE =  2.25699E-07   DIIS = 7
CCSD Iteration   9: CCSD correlation = -0.153512689430021   dE = -5.98074E-08   DIIS = 7
CCSD Iteration  10: CCSD correlation = -0.153512693468796   dE = -4.03877E-09   DIIS = 7
CCSD Iteration  11: CCSD c

-0.1535126969598728

In [9]:
check=True

if check==True:
    A.t1 = np.random.rand(*A.t1.shape)
    A.t2 = np.random.rand(*A.t2.shape)

A.compute_energy()

CCSD Iteration   0: CCSD correlation = 1.365876598072616   dE = -1.36588E+00   MP2
CCSD Iteration   1: CCSD correlation = 0.902364293977967   dE = -4.63512E-01   DIIS = 0
CCSD Iteration   2: CCSD correlation = 1.397487177548195   dE =  4.95123E-01   DIIS = 1
CCSD Iteration   3: CCSD correlation = 1.586696725313544   dE =  1.89210E-01   DIIS = 2
CCSD Iteration   4: CCSD correlation = 1.069906349733953   dE = -5.16790E-01   DIIS = 3
CCSD Iteration   5: CCSD correlation = 0.865993810573281   dE = -2.03913E-01   DIIS = 4
CCSD Iteration   6: CCSD correlation = 0.942018788981227   dE =  7.60250E-02   DIIS = 5
CCSD Iteration   7: CCSD correlation = 0.909220811511640   dE = -3.27980E-02   DIIS = 6
CCSD Iteration   8: CCSD correlation = 0.477018795619668   dE = -4.32202E-01   DIIS = 7
CCSD Iteration   9: CCSD correlation = 0.587770265179010   dE =  1.10751E-01   DIIS = 7
CCSD Iteration  10: CCSD correlation = 0.527164530479491   dE = -6.06057E-02   DIIS = 7
CCSD Iteration  11: CCSD correlation 

-0.15351269419713987

# Correlation energy

In [ ]:
A.FinalEnergy

# T1-amplitudes

In [ ]:

A.t1

# T2-amplitudes

In [ ]:
A.t2